In [ ]:
!pip install cerberus gamspy_base


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [ ]:
from pathlib import Path
from gamspy_base import directory
from gams.connect import ConnectDatabase

First, I ran xl2times on Demo 1 using the following command:
```
cd ../xl2times
python utils/run_benchmarks.py --times_dir /scratch/htc/skrishna/xl2times/TIMES_model/ benchmarks.yml --dd --run DemoS_001-all
```

In [ ]:
!ls ../xl2times/benchmarks/out/DemoS_001-all/

diffile.gdx	   raw_tables.txt		    scenario.run
gams.opt	   runmodel.gms			    source
__init__.py	   runmodel.lst			    stdout
merged_tables.txt  scenario~Data_250808_112629.gdx  ts.dd
milestonyr.dd	   scenario.gdx
output.dd	   scenario.lst


In [7]:
demo_name = "DemoS_001-all"
demo_path = f"../xl2times/benchmarks/out/{demo_name}"
gdx_path = f"{demo_path}/scenario.gdx"
sqlite_file = f"{demo_name}.db3"

In [8]:
# Read the created GDX file and convert it to SQLite
cdb = ConnectDatabase(system_directory=directory)
# Read the GDX file
cdb.execute({
    'GDXReader': {
        'file': gdx_path,
        'symbols': 'all'
        }})
# Check if the sqlite file already exists and delete it if it does
if Path(sqlite_file).exists():
    print(f"Deleting existing SQLite file: {sqlite_file}")
    Path(sqlite_file).unlink()
# Create the SQLite database
cdb.execute({
    'SQLWriter': {
        'connection': {'database': sqlite_file},
        'ifExists': 'replace',
        'connectionType': 'sqlite',
        'symbols': 'all',
        'skipText': True,
        }})

In [9]:
!julia --project=. demo.jl

]0;Julia]0;JuliaERROR: LoadError: SQLite.SQLiteException("no such column: ALL_REG")
Stacktrace:
  [1] sqliteerror(args::SQLite.DB)
    @ SQLite ~/.julia/packages/SQLite/UqCGE/src/SQLite.jl:34
  [2] macro expansion
    @ ~/.julia/packages/SQLite/UqCGE/src/base.jl:10 [inlined]
  [3] prepare_stmt_wrapper
    @ ~/.julia/packages/SQLite/UqCGE/src/SQLite.jl:110 [inlined]
  [4] SQLite.Stmt(db::SQLite.DB, sql::String; register::Bool)
    @ SQLite ~/.julia/packages/SQLite/UqCGE/src/SQLite.jl:147
  [5] Stmt
    @ ~/.julia/packages/SQLite/UqCGE/src/SQLite.jl:146 [inlined]
  [6] prepare
    @ ~/.julia/packages/SQLite/UqCGE/src/SQLite.jl:181 [inlined]
  [7] execute
    @ ~/.julia/packages/DBInterface/nQcsk/src/DBInterface.jl:130 [inlined]
  [8] execute
    @ ~/.julia/packages/DBInterface/nQcsk/src/DBInterface.jl:152 [inlined]
  [9] (::TIMES.var"#1#2"{SQLite.DB})(::Pair{String, String})
    @ TIMES ./none:0
 [10] iterate(g::Base.Generator{Dict{String, String}, TIMES.var"#1#2"{SQLite.DB}}, s::Int

In [14]:
%%bash
# This is an interesting error because the set ALL_REG exists in both the output DD file and in the scenario.gdx file
grep -i all_reg ../xl2times/benchmarks/out/DemoS_001-all/*.dd

../xl2times/benchmarks/out/DemoS_001-all/output.dd:SET ALL_REG


In [16]:
!/scratch/htc/skrishna/gams/gams50.3_linux_x64_64_sfx/gdxdump ../xl2times/benchmarks/out/DemoS_001-all/scenario.gdx | grep -i all_reg | head

Set ALL_REG(*) External + Internal Regions /
Set REG(ALL_REG) Region /
Alias (ALL_R, ALL_REG);
Set PRC_TS(ALL_REG,PRC,ALL_TS) Timeslices for a process / /;
Set REG_RMAP(REG_GRP,ALL_REG) 'Grouping of regions in/out of area of study' / /;
Set TS_GROUP(ALL_REG,TSLVL,TS) Timeslice Level assignment /
Set TS_MAP(ALL_REG,ALL_TS,ALL_TS) Timeslice hierarchy tree: node+below / /;
Set TOP_IRE(ALL_REG,COM,ALL_REG,COM,PRC) Trade within area of study /
Set PRC_PKNO(ALL_REG,PRC) Processes which cannot be involved in peaking / /;
Set PRC_PKAF(ALL_REG,PRC) Flag for default value of NCAP_PKCNT / /;
